### Scrape Web pages for particular title

In [ ]:
!pip install llama-index beautifulsoup4 requests duckdb nltk wordcloud matplotlib seaborn


In [ ]:
import requests
from bs4 import BeautifulSoup
import requests
import os
from llama_index.core import GPTVectorStoreIndex, SimpleDirectoryReader, ListIndex, Document
from google.colab import userdata
#from dotenv import load_dotenv
openai_api_key=userdata.get('openai_api_key');
##load_dotenv()
##openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
def scrape_job(job_title):
    url = f"https://www.jobly.fi/tyopaikat?search={job_title}" # Construct the search URL
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for bad status codes

    soup = BeautifulSoup(response.content, "html.parser") # Parse the HTML content
    job_listings = soup.find_all("a", class_="recruiter-job-link") # Example selector: Replace with the actual class or id of the job listing container.

    job_links = [listing['href'] for listing in job_listings]
    top3_jobs=job_links[::2][:3]
    #print(top3_jobs)
    return top3_jobs


In [ ]:
job_title="Data Scientist"
top3_jobs=scrape_job(job_title)
top3_jobs

['https://www.jobly.fi/tyopaikka/analyst-data-scientist-2307774',
 'https://www.jobly.fi/tyopaikka/full-stack-data-scientist-2299989',
 'https://www.jobly.fi/tyopaikka/summer-trainee-data-scientist-trainee-decision-support-and-analytics-2252120']

In [ ]:
def job_description(scrape_result):
    dic = {}
    for i, link in enumerate(top3_jobs):
      response = requests.get(link)
      soup = BeautifulSoup(response.content, "html.parser")
      job_description_element = soup.find("div", class_="l-main")
      job_description = job_description_element.get_text(strip=True)
      job_description = job_description_element.get_text(strip=True)
      text = "Hae paikkaaTallenna työpaikka"
      first_occurrence = job_description.find(text)
      second_occurrence = job_description.find(text, first_occurrence + 1)
      extracted_text = job_description[first_occurrence+len(text):second_occurrence]

      dic[i] = extracted_text
    return dic

In [ ]:
summary_3jobs = job_description(top3_jobs)

In [ ]:
#summary_3jobs

### If the text is in Finnish, then translate it to English

In [ ]:
#!pip install python-dotenv

In [ ]:
def semantic_search(query):
    insights = {}
    for item, summary in summary_3jobs.items():
      # Create a document object
      documents = [Document(text=summary)]
      index = GPTVectorStoreIndex.from_documents(documents)
      # Get a QueryEngine object from the index
      query_engine = index.as_query_engine()
      # Use the QueryEngine to query the index
      response = query_engine.query(query)
      insights[item] = response.response
    return insights


In [ ]:
prompt = "Describe in 4 paragraphs: the domain/team and the seniority level in the company, the job role/key responsibilities, the qualifications needed for the job, the key skills required for the job (both technical and soft skills). "
result=semantic_search(prompt)
result

{0: 'The domain of the team is Product & Price Liability within the insurance company, focusing on developing and pricing liability insurances for commercial customers across the Nordic region. The team works collaboratively to create the best solutions and improve pricing strategies. The seniority level in the company for this role is positioned as an Analyst, responsible for utilizing data analysis and modern tools to enhance pricing accuracy and profitability while ensuring customer satisfaction and peace of mind.\n\nThe key responsibilities of the job role include analyzing historical and external data to improve processes, enhance data quality, and refine pricing models for liability insurances. The Analyst will play a crucial role in setting risk-correct prices for a large customer base, impacting sales processes and collaborating with other units such as sales and claims. The role involves taking ownership of the common mission to grow profitably and establish the company as a c

In [ ]:
result = {0: 'The domain of the team is Product & Price Liability within the insurance company, focusing on developing and pricing liability insurances for commercial customers across the Nordic region. The team works collaboratively to create the best solutions and improve pricing strategies. The seniority level in the company for this role is positioned as an Analyst, responsible for utilizing data analysis and modern tools to enhance pricing accuracy and profitability while ensuring customer satisfaction and peace of mind.\n\nThe key responsibilities of the job role include analyzing historical and external data to improve processes, enhance data quality, and refine pricing models for liability insurances. The Analyst will play a crucial role in setting risk-correct prices for a large customer base, impacting sales processes and collaborating with other units such as sales and claims. The role involves taking ownership of the common mission to grow profitably and establish the company as a caring insurance provider.\n\nQualifications needed for the job include an academic background in mathematics, statistics, physics, engineering, or computer science. Fluent communication skills in English and Finnish are required, along with previous experience in analytical roles being beneficial. The role does not mandate prior experience but values a passion for data science, statistics, and problem-solving, as well as a willingness to learn and adapt to new tools and technologies used within the team.\n\nKey skills required for the job encompass technical proficiency in tools like Databricks, Python, SQL, and Excel, as well as a strong analytical mindset and business acumen. Soft skills such as efficient work ethic, patience in data analysis, willingness to share knowledge, and a collaborative attitude towards achieving common goals are highly valued. The ideal candidate should be passionate about leveraging data insights to drive valuable solutions, dedicated to quality checking data, and enthusiastic about exploring innovative approaches to pricing and risk assessment in the insurance domain.',
 1: "The domain/team within Nokia Technologies is the Patent Analytics team, which plays a crucial role in managing the company's patent portfolio by utilizing advanced analytics to derive strategic insights. The team consists of data scientists, patent attorneys, and business strategists who collaborate to develop innovative solutions that impact decision-making processes. This team operates in a dynamic and interdisciplinary environment that encourages creativity, adaptability, and exploration of cutting-edge technologies to drive continuous growth and development.\n\nThe job role for this position is a Full Stack Data Scientist within the Patent Analytics team at Nokia Technologies. The key responsibilities include developing and deploying advanced AI and Generative AI models to extract insights from complex patent data, designing and implementing scalable ETL pipelines, building end-to-end solutions including cloud-based applications and visualization dashboards, collaborating with cross-functional teams for seamless integration of data science solutions, and communicating findings through compelling data visualizations to technical and non-technical stakeholders.\n\nThe qualifications required for this job include a Ph.D. or M.Sc. degree in a relevant field such as Data Science, Computer Science, Statistics, Mathematics, or Engineering, along with at least 5 years of experience as a data scientist focusing on patent analytics. Strong programming skills in languages like Python, experience with big data tools such as Spark, and proficiency in advanced statistical methods are essential. Additionally, familiarity with SQL, NoSQL databases, cloud-based data platforms, and expertise in supervised, unsupervised, and deep learning methods, particularly Generative AI, are necessary qualifications.\n\nThe key skills required for this role encompass both technical and soft skills. Technical skills include proficiency in programming languages, experience with big data tools, expertise in statistical methods and AI models, and the ability to create interactive dashboards and reports. Soft skills such as excellent communication skills to convey technical concepts, strong interpersonal skills to collaborate effectively within a global team, and a proactive approach to leveraging technology for impactful results are also crucial for success in this position.",
 2: "The domain/team for this position is Decision Support and Analytics within UPM Fibres, a department focused on advancing UPM's Beyond Fossils strategy. The team is led by Sauli Järvenpää, who is the head of Decision Support and Analytics. This team plays a crucial role in influencing decisions across UPM Fibres' businesses, utilizing data science and analytics to provide insightful conclusions and forecasts to decision-makers.\n\nThe job role entails studying and determining business problems, selecting appropriate methodologies for analysis, analyzing data, and presenting conclusions to decision-makers. Additionally, the role involves collaborating with business experts to develop performance metrics, as well as participating in the development and deployment of decision support tools based on machine learning, simulation, and optimization. The position also involves expanding the business's capacity in utilizing generative AI, with a specific focus on large language models (LLMs).\n\nTo qualify for this position, candidates are required to be Master's students in an operations research field such as applied math, statistics, or computer science, with a strong interest in data science, programming, optimization, or simulation. The role also values candidates who demonstrate curiosity, excellent teamwork skills, and a proactive approach. Proficiency in English is essential for effective communication within the international company environment.\n\nKey technical skills required for the job include expertise in data analysis, methodology selection, and the development of decision support tools using machine learning and optimization techniques. Soft skills such as teamwork, curiosity, and initiative-taking are highly valued for this role. Effective communication skills in English are crucial for collaborating with colleagues and presenting findings to decision-makers. The ability to adapt to a dynamic work environment and a willingness to learn and grow within the organization are also important attributes for success in this position."}

In [ ]:
dix = {0: 'The job role entails working as a Data Scientist at Wärtsilä Voyage, focusing on the global Vessel Traffic Services (VTS) and Port Management Information Systems (PMIS) product lines. The key responsibilities include conceptualizing and specifying solution requirements in collaboration with business stakeholders, formalizing tasks, researching different approaches, designing and developing machine learning applications and algorithms, and building and optimizing machine learning pipelines. The role involves utilizing data from various sources to develop AI services that enhance safety, decarbonization, and operational efficiency in the maritime industry.\n\nTo qualify for the position, candidates should have a strong mathematical background in statistics and understanding of classical machine learning algorithms. Proficiency in software development using Python, familiarity with machine learning stack (NumPy, Pandas, Scikit-Learn, Pytorch), and web application stack (Fastapi/Flask) are essential. Additionally, experience with geospatial data, data engineering, and Azure cloud services is advantageous. Candidates with experience in the maritime industry, particularly in port operations or maritime logistics, and knowledge of supply chain and logistics practices are preferred.\n\nThe key skills required for the job encompass a blend of technical expertise and soft skills. Technical skills include proficiency in machine learning algorithms, software development in Python, and familiarity with relevant tools and technologies like NumPy, Pandas, Scikit-Learn, Pytorch, Fastapi/Flask, and Azure cloud services. Strong problem-solving abilities, the capacity to communicate abstract concepts to technical and non-technical stakeholders, and experience in the maritime industry are valuable soft skills for this role.\n\nThe domain of this job role is focused on the maritime industry, specifically Vessel Traffic Services (VTS) and Port Management Information Systems (PMIS). The position requires a deep understanding of maritime operations, port logistics, and supply chain practices. The role involves leveraging data from navigation, port schedules, weather, and other sources to develop AI services that enhance safety, decarbonization, and operational efficiency in the maritime sector.\n\nIn terms of seniority level, the position of Data Scientist at Wärtsilä Voyage appears to be at an intermediate to senior level. The job requires a combination of technical expertise, industry knowledge, and the ability to collaborate with business stakeholders to drive innovation in technology and services within the maritime industry. The role involves significant responsibilities in developing machine learning applications, optimizing pipelines, and contributing to the transformation towards a cleaner, more sustainable future in the marine and energy sectors.',
 1: "The job role entails supporting the implementation of the Data & AI strategy by creating, implementing, and maintaining AI solutions in collaboration with stakeholders across various business domains. This involves working with data engineers to manage data ingestion and migration, building and testing data models, deploying ML models in production, and supporting knowledge building across the organization. The role also includes ensuring compliance with the EU AI Act and driving the citizen community within the company.\n\nQualifications for the job include a master's degree in AI, statistics, computer science, or engineering, along with a minimum of 5 years of experience in data science projects. Experience in developing statistical and machine learning solutions for business problems, particularly in the context of manufacturing processes, is essential. Familiarity with relational databases, SQL-like query languages, cloud technologies like MS Azure, and tools such as Databricks is required. Knowledge of non-relational database technology and distributed computing is beneficial.\n\nKey technical skills required for the job include proficiency in developing statistical and machine learning solutions, experience with data ingestion and migration, and knowledge of cloud technologies for AI solution delivery. Soft skills such as effective communication with business stakeholders, collaboration with cross-functional teams, and the ability to drive knowledge sharing initiatives are crucial. The role demands a proactive approach to problem-solving, attention to detail in data analysis, and the ability to translate complex technical concepts into actionable insights for stakeholders.\n\nThe domain of the job role is within Data & Analytics, specifically focusing on implementing AI solutions to address process challenges in areas like plant operations, R&D, sales, and workplace productivity. The seniority level of the position is at an experienced level, requiring a minimum of 5 years of relevant working experience in data science projects. The role involves working closely with stakeholders, data engineers, and cloud experts to deliver AI solutions that drive business value and ensure compliance with regulations.\n\nIn summary, the job role involves leveraging data and AI technologies to address business challenges across different domains within the organization. The qualifications needed include a master's degree in a relevant field and substantial experience in data science projects, particularly in manufacturing processes. Key technical skills encompass proficiency in developing data models, deploying ML solutions, and working with cloud technologies, while essential soft skills include effective communication, collaboration, and a proactive problem-solving approach. The domain of the role is Data & Analytics, and the seniority level is at an experienced level, requiring a solid foundation in data science and AI technologies.",
 2: "The job role of the Senior Scientist at NordGen Farm Animals involves leading and participating in projects related to the conservation and sustainable use of farm animal genetic resources. The key responsibilities include strengthening the scientific team's capacity to achieve NordGen's strategic goals, leading research projects, facilitating networks, giving lectures, and communicating with stakeholders. The Senior Scientist will also be responsible for identifying new project opportunities to enhance conservation efforts and promoting Nordic animal genetic resources.\n\nQualifications required for the job include a Ph.D. in animal genetics or related fields, relevant working experience in research, breeding, or conservation, successful project planning and management experience, a high number of published scientific articles, and experience in international research cooperation. Fluency in one of the Scandinavian languages and English is essential. Additional merits include experience in genomics data analysis, database utilization, a wide network of collaboration partners, and Nordic collaboration experience.\n\nThe key skills required for the job encompass both technical and soft skills. Technical skills include expertise in animal genetics, research methodologies, project management, and data analysis in genomics. Soft skills consist of being a team player, working independently, innovative thinking, adaptability to new challenges, collaboration with diverse organizations, and quick decision-making abilities. The Senior Scientist should also possess excellent communication skills to engage with stakeholders effectively.\n\nThe domain of this job is farm animal genetics, conservation, and sustainable use of genetic resources. The Senior Scientist will be working within the field of animal genetics, focusing on the conservation and utilization of Nordic farm animal genetic resources. The role involves contributing to the scientific advancements in the field and promoting sustainable practices in farm animal breeding and conservation efforts.\n\nThis position is at a senior level, requiring a Ph.D. and significant experience in research, breeding, or conservation. The Senior Scientist will be leading projects, consulting on conservation strategies, and engaging with stakeholders at national and international levels. The role demands a high level of expertise in the domain of farm animal genetics and a proactive approach to driving sustainable conservation initiatives within the Nordic countries and beyond."}

In [ ]:
### Use prompt again to get hard skills, soft skills etc. Or use LLMs from hugging face for free. Or use Azure cognitive search for knowledge mining, LLM Reasoning, Contextual Knowledge

In [ ]:
from transformers import pipeline
def get_insights(summary_dict):
  company_insights = {}
  llm_qa = pipeline("question-answering")
  llm_summarize  = pipeline("summarization", model="facebook/bart-large-cnn")
  question1 = "What is the name of the company/team?"
  question2 = "What is the domain of the company?"
  question3 = "What is the seniority level?"
  question4 = "Answer the keywords separated by commas."
  for index in summary_dict:
    q_context1 = summary_dict[index].split('\n\n')[0]
    q_context2 = summary_dict[index].split('\n\n')[1]
    q_context3 = summary_dict[index].split('\n\n')[-1]
    #q_context = q_context1 + q_context2 + q_context3
    company_name = llm_qa(question=question1, context=q_context1)
    domain = llm_qa(question=question1, context=q_context2)
    seniority = llm_qa(question=question1, context=q_context1)
    #skills = llm_qa(question = question4, context=q_context3)
    responsibility = llm_summarize(summary_dict[index].split('\n\n')[0], max_length=115, clean_up_tokenization_spaces=True)[0]['summary_text']
    qualification = llm_summarize(summary_dict[index].split('\n\n')[2], max_length=115, clean_up_tokenization_spaces=True)[0]['summary_text']
    skills = llm_summarize(summary_dict[index].split('\n\n')[-1], max_length=115, clean_up_tokenization_spaces=True)[0]['summary_text'] # should you further pass it to qa, or directly put it in qa ?
    company_insights[index] = {'company_name': company_name, 'responsibility': responsibility, 'qualification': qualification, 'skills': skills, 'domain':domain,'seniority':seniority}
  return company_insights


In [ ]:
insights = get_insights(result)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Device set to use cpu
Your max_length is set to 115, but your input_length is only 88. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=44)
Your max_length is set to 115, but your input_length is only 88. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=44)
Your max_length is set to 115, but your input_length is only 104. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider d

In [ ]:
insights

{0: {'company_name': {'score': 0.9926282167434692,
   'start': 26,
   'end': 51,
   'answer': 'Product & Price Liability'},
  'responsibility': 'The team works collaboratively to create the best solutions and improve pricing strategies. The domain of the team is Product & Price Liability within the insurance company. The seniority level in the company for this role is positioned as an Analyst, responsible for utilizing data analysis and modern tools to enhance pricing accuracy and profitability.',
  'qualification': 'Fluent communication skills in English and Finnish are required, along with previous experience in analytical roles. The role does not mandate prior experience but values a passion for data science, statistics, and problem-solving. Qualifications needed for the job include an academic background in mathematics,Statistics, physics, engineering, or computer science.',
  'skills': 'Key skills required for the job encompass technical proficiency in tools like Databricks, Pytho

In [ ]:
import pandas as pd
# Flatten dictionary into a DataFrame
df = pd.DataFrame.from_dict(insights, orient='index')

# Extract nested fields (flattening the structure)
df['company_name'] = df['company_name'].apply(lambda x: x.get('answer') if isinstance(x, dict) else x)
df['domain'] = df['domain'].apply(lambda x: x.get('answer') if isinstance(x, dict) else x)
df['seniority'] = df['seniority'].apply(lambda x: x.get('answer') if isinstance(x, dict) else x)

# Display DataFrame
df

,company_name,responsibility,qualification,skills,domain,seniority
0,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage
1,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI
2,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals


In [ ]:
# Ensure the length of top3_jobs matches the number of rows in your DataFrame
if len(top3_jobs) == len(df):
    df['Top3_Jobs'] = top3_jobs

df


,company_name,responsibility,qualification,skills,domain,seniority,Top3_Jobs
0,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage,https://www.jobly.fi/tyopaikka/analyst-data-sc...
1,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI,https://www.jobly.fi/tyopaikka/full-stack-data...
2,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals,https://www.jobly.fi/tyopaikka/summer-trainee-...


In [ ]:
#df.Top3_Jobs.values

In [ ]:
insights = get_insights(dix)

df1 = pd.DataFrame.from_dict(insights, orient='index')

# Extract nested fields (flattening the structure)
df1['company_name'] = df['company_name'].apply(lambda x: x.get('answer') if isinstance(x, dict) else x)
df1['domain'] = df['domain'].apply(lambda x: x.get('answer') if isinstance(x, dict) else x)
df1['seniority'] = df['seniority'].apply(lambda x: x.get('answer') if isinstance(x, dict) else x)
top3_jobs_df1 = [[] for _ in range(len(df1))]
df1['Top3_Jobs'] = top3_jobs_df1

# Display DataFrame
df1

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Device set to use cpu
Your max_length is set to 115, but your input_length is only 113. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=56)
Your max_length is set to 115, but your input_length is only 102. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=51)
Your max_length is set to 115, but your input_length is only 98. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider 

,company_name,responsibility,qualification,skills,domain,seniority,Top3_Jobs
0,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage,[]
1,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI,[]
2,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals,[]


In [ ]:
# Append rows of df1 to df
df = pd.concat([df, df1], ignore_index=True)

# Display the updated DataFrame
df


,company_name,responsibility,qualification,skills,domain,seniority,Top3_Jobs
0,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage,https://www.jobly.fi/tyopaikka/analyst-data-sc...
1,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI,https://www.jobly.fi/tyopaikka/full-stack-data...
2,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals,https://www.jobly.fi/tyopaikka/summer-trainee-...
3,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage,[]
4,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI,[]
5,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals,[]


In [ ]:
df.to_csv('job_data.csv', index=False)

In [ ]:
import pandas as pd
df = pd.read_csv('job_data.csv')
df

,company_name,responsibility,qualification,skills,domain,seniority,Top3_Jobs
0,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage,https://www.jobly.fi/tyopaikka/analyst-data-sc...
1,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI,https://www.jobly.fi/tyopaikka/full-stack-data...
2,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals,https://www.jobly.fi/tyopaikka/summer-trainee-...
3,Data Scientist at Wärtsilä Voyage,The job role entails working as a Data Scienti...,Key skills include proficiency in machine lear...,Data Scientist at Wärtsilä Voyage appears to b...,Fastapi/Flask,Data Scientist at Wärtsilä Voyage,[]
4,Data & AI,The job role entails supporting the implementa...,Key technical skills required for the job incl...,The role involves leveraging data and AI techn...,engineering,Data & AI,[]
5,NordGen Farm Animals,Senior Scientist at NordGen Farm Animals leads...,The Senior Scientist should possess excellent ...,"The Senior Scientist will be leading projects,...",collaboration partners,NordGen Farm Animals,[]


In [ ]:
import numpy as np
np.savetxt(r'np.txt', df.values, fmt='%s')


In [ ]:
import duckdb, pandas as pd

# Connect to an in-memory DuckDB database
conn = duckdb.connect(database=":memory:", read_only=False)

conn.execute("""
    CREATE TABLE IF NOT EXISTS job_data (
        company_name TEXT,
        responsibility TEXT,
        qualification TEXT,
        skills TEXT,
        domain TEXT,
        seniority TEXT,
        links Text
    )
""")

# Insert data manually
for _, row in df.iterrows():
    conn.execute("INSERT INTO job_data VALUES (?, ?, ?, ?, ?, ?, ?)",
                 (row['company_name'], row['responsibility'], row['qualification'],
                  row['skills'], row['domain'], row['seniority'], row['Top3_Jobs']))




In [ ]:
# Verify stored data
result = conn.execute("SELECT * FROM job_data").fetchdf()
print(result)

                        company_name  \
0  Data Scientist at Wärtsilä Voyage   
1                          Data & AI   
2               NordGen Farm Animals   
3  Data Scientist at Wärtsilä Voyage   
4                          Data & AI   
5               NordGen Farm Animals   

                                      responsibility  \
0  The job role entails working as a Data Scienti...   
1  The job role entails supporting the implementa...   
2  Senior Scientist at NordGen Farm Animals leads...   
3  The job role entails working as a Data Scienti...   
4  The job role entails supporting the implementa...   
5  Senior Scientist at NordGen Farm Animals leads...   

                                       qualification  \
0  Key skills include proficiency in machine lear...   
1  Key technical skills required for the job incl...   
2  The Senior Scientist should possess excellent ...   
3  Key skills include proficiency in machine lear...   
4  Key technical skills required for the job i

From here, what I can do is use the previously developed application to get comma separated list for responsibility, qualification and skills. Then it gets ready for the wordcloud visuaization  

In [ ]:
# Mount google drive in colab
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
drive_path = "/content/drive/MyDrive/job_data.parquet"


In [ ]:
# Save the table

import os

# Check if the file already exists
if os.path.exists(drive_path):
    # Load existing job data
    existing_df = pd.read_parquet(drive_path)

    # Append new jobs that are not already in the dataset
    updated_df = pd.concat([existing_df, result]).drop_duplicates() # how to simulate delta lake instead ?

    # Save updated dataset
    updated_df.to_parquet(drive_path, index=False)

    print(f"Updated job dataset saved in Google Drive: {drive_path}")
else:
    # Save first-time dataset
    result.to_parquet(drive_path, index=False)
    print(f"Job dataset created and saved in Google Drive: {drive_path}")


Job dataset created and saved in Google Drive: /content/drive/MyDrive/job_data.parquet


In [ ]:
!cat drive/MyDrive/job_data.parquet

 oE�!{�toU� c.efforts. ,6 (�The job role entails working as a Data Scientist at Wärtsilä Voyage, focusing on the global Vessel Traffic Services (VTS) and Port Management Information Systems (PMIS) product lines. The role involves utilizing data from various sources to develop AI services that enhance safety, decarbonization, and operational efficiency.�Senior Scientist at NordGen Farm Animals leads and participates in projects related to the conservation and sustainable use of farm animal genetic resources. Key responsibilities include strengthening the scientific team's capacity to achieve NordGen's strategic goals. The Senior Scientist will also be responsible for identifying new project opportunities to enhance conservation efforts.   
:q�E�tisA�� animal genetics, research methodologies,!X(ject managea�, A42H!Ygenomic:3cons�of beA| a%�0 player, workPindependently, innovaEcthin#, adapt=�new cha!5 ga�JcTdiverse organizations. ,6 (�T

Also, saving it in duckdb format

In [ ]:
duckdb_path = "/content/drive/MyDrive/job_data.duckdb"


In [ ]:
'''
# Connect to DuckDB database stored in Google Drive
conn = duckdb.connect(database=duckdb_path, read_only=False)

# Create table if it does not exist
# Added 'links TEXT' column to match the DataFrame structure
conn.execute("""
    CREATE TABLE IF NOT EXISTS job_data (
        company_name TEXT,
        responsibility TEXT,
        qualification TEXT,
        skills TEXT,
        domain TEXT,
        seniority TEXT,
        links TEXT  -- Added this column
    )
""")

# Register the DataFrame as a view in the current connection
conn.register('df_view', df) # Register the Pandas DataFrame as a view in this connection.

# Append only new records that don’t already exist
conn.execute("""
    INSERT INTO job_data
    SELECT * FROM df_view
    WHERE NOT EXISTS (
        SELECT 1 FROM job_data WHERE job_data.company_name = df_view.company_name
    )
""")

# Verify stored data
updated_result = conn.execute("SELECT * FROM job_data").fetchdf()
print(updated_result)

# Close connection
conn.close()'''

'\n# Connect to DuckDB database stored in Google Drive\nconn = duckdb.connect(database=duckdb_path, read_only=False)\n\n# Create table if it does not exist\n# Added \'links TEXT\' column to match the DataFrame structure\nconn.execute("""\n    CREATE TABLE IF NOT EXISTS job_data (\n        company_name TEXT,\n        responsibility TEXT,\n        qualification TEXT,\n        skills TEXT,\n        domain TEXT,\n        seniority TEXT,\n        links TEXT  -- Added this column \n    )\n""")\n\n# Register the DataFrame as a view in the current connection\nconn.register(\'df_view\', df) # Register the Pandas DataFrame as a view in this connection.\n\n# Append only new records that don’t already exist\nconn.execute("""\n    INSERT INTO job_data\n    SELECT * FROM df_view\n    WHERE NOT EXISTS (\n        SELECT 1 FROM job_data WHERE job_data.company_name = df_view.company_name\n    )\n""")\n\n# Verify stored data\nupdated_result = conn.execute("SELECT * FROM job_data").fetchdf()\nprint(update

### Specify the domain / field of industry

### Get all the seniority level skillsets required

### Summarize the task and skillset

### Highlight the top keywords / skillsets

### What are the other keywords that you might search for your similar positions

### Report the skills missing and visualize in Venn diagram, bubble diagram etc

### Suggest how to upskill based on current market